# Telco Customer Churn Prediction

This project builds a machine learning model to predict telecom customer churn using customer demographics, contract details, and service usage data.

## Objectives
- Clean and preprocess customer data
- Perform feature engineering
- Train a classification model
- Evaluate model performance
- Identify key drivers of churn

# Import libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

# Load Dataset ( Markdown + code)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
path = "/content/drive/MyDrive/github_repo/telco-churn-ml-pipeline/data/Telco_Customer_Churn.csv"
df =pd.read_csv(path)

In [6]:
df.shape

(7043, 21)

# Fix the TotalCharges column

In [8]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [9]:
df["TotalCharges"]= pd.to_numeric(df["TotalCharges"], errors= "coerce")

In [10]:
# Filling missing values

In [11]:
df["TotalCharges"].fillna(df["TotalCharges"].median(),inplace=True)

/tmp/ipykernel_302/1795292956.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(),inplace=True)


In [12]:
df["Churn"] = df["Churn"].map({"Yes":1,"No":0})

In [13]:
df["Churn"].value_counts()

,count
Churn,
0,5174
1,1869


# Drop ID column

In [14]:
df =df.drop("customerID", axis=1)

# Convert categorical features

In [15]:
df_encoded = pd.get_dummies(df, drop_first= True)

In [16]:
df_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


# Convert boolean columns to integers

In [17]:
df_encoded = df_encoded.astype(int)

In [18]:
df_encoded.dtypes

,0
SeniorCitizen,int64
tenure,int64
MonthlyCharges,int64
TotalCharges,int64
Churn,int64
gender_Male,int64
Partner_Yes,int64
Dependents_Yes,int64
PhoneService_Yes,int64
MultipleLines_No phone service,int64


# Split data

In [20]:
from sklearn.model_selection import train_test_split
X =df_encoded.drop("Churn", axis=1)
y =df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size= 0.2, random_state=42, stratify=y)

In [21]:
print(X_train.shape)
print(X_test.shape)

(5634, 30)
(1409, 30)


# Train Model

In [22]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
y_pred = model.predict(X_test)
y_prob= model.predict_proba(X_test)[:,1]
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("ROC AUC", roc_auc_score(y_test, y_prob))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))


Accuracy:  0.7899219304471257
ROC AUC 0.8194192565036557
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

[[926 109]
 [187 187]]


# Feature Importance

In [26]:
import pandas as pd
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False, inplace=True)
importance.head(10)

,0
TotalCharges,0.190727
tenure,0.181784
MonthlyCharges,0.133352
PaymentMethod_Electronic check,0.042992
InternetService_Fiber optic,0.039229
gender_Male,0.031135
Contract_Two year,0.030598
OnlineSecurity_Yes,0.029206
PaperlessBilling_Yes,0.028160
Partner_Yes,0.025237


In [27]:
import joblib
joblib.dump(model, "random_forest.pkl")
print ("Model saved successfully")

Model saved successfully


In [29]:
import os
os.listdir()

['.config', 'drive', 'random_forest.pkl', 'sample_data']

In [31]:
import joblib
joblib.dump(
    model,
    "/content/drive/MyDrive/github_repo/telco-churn-ml-pipeline/models/random_forest.pkl"
)

print("Model saved to Google Drive")

Model saved to Google Drive


In [32]:
os.listdir("/content/drive/MyDrive/github_repo/telco-churn-ml-pipeline/models")

['random_forest.pkl']

In [33]:
import joblib

# Load the saved model
loaded_model = joblib.load("/content/drive/MyDrive/github_repo/telco-churn-ml-pipeline/models/random_forest.pkl")

# Test prediction on one record from your test set
sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)
probability = loaded_model.predict_proba(sample)

print("Prediction:", prediction)
print("Probability:", probability)

Prediction: [0]
Probability: [[1. 0.]]


In [35]:
import os
# create output files if does not  exist
os.makedirs("/content/drive/MyDrive/github_repo/telco-churn-ml-pipeline/outputs", exist_ok=True)
# save feature importance
feature = importance.to_csv("/content/drive/MyDrive/github_repo/telco-churn-ml-pipeline/outputs/feature_importance.csv")
print("Features importance saved")

Features importance saved
